# Customer Intelligence Platform — Notebook 2: Dimensionality Reduction & Visualization
**Portfolio Project | Notebook 3 of 5**

---

## Learning Objectives
1. Reuse Notebook 1's *fitted* imputer/scaler (not refit new ones) to project the same feature space used for clustering — consistency matters even for a diagnostic step
2. Apply PCA, t-SNE, and UMAP to visualize the 10-dimensional segmentation in 2D, and judge which technique communicates the segment structure most faithfully
3. Understand — and be able to explain in an interview — **why none of these three techniques should ever feed back into the clustering or churn model as an input feature** in this project's design

> **Senior engineer framing:** This project's README states plainly: *"PCA and UMAP are applied purely for diagnostic visualization... never as a modeling input."* That's a deliberate architectural decision, not an oversight — it keeps the churn model's feature space fully interpretable (a stakeholder can ask "why did the model flag this customer?" and get an answer in terms of real business quantities, not an opaque embedding dimension). This notebook exists to *validate* the segmentation from Notebook 1 visually and to build the kind of 2D scatter plot that goes directly into a stakeholder deck — not to engineer new features.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("umap-learn not installed — run `pip install umap-learn` to enable the UMAP sections.")

sns.set_theme(style="whitegrid", palette="deep")
RNG_SEED = 42

DATA_PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")

df = pd.read_csv(DATA_PROCESSED_DIR / "customers_with_segments.csv")
artifacts = joblib.load(MODELS_DIR / "segmentation_artifacts.joblib")
imputer, scaler, feature_cols = artifacts["imputer"], artifacts["scaler"], artifacts["feature_cols"]

df.shape

---
## Part 1: Rebuild the Exact Feature Space Used for Clustering

### TODO 1 — Transform (don't refit!) using the saved imputer and scaler

**HINT:** call `.transform()`, not `.fit_transform()` — the whole point of persisting these artefacts in Notebook 1 was so every later notebook uses the *identical* imputation values and scaling parameters. Refitting here would silently drift the feature space out of sync with the segmentation.

In [ ]:
X = df[feature_cols].copy()

# TODO: apply imputer.transform then scaler.transform (NOT fit_transform)
X_imputed = ...
X_scaled = ...

X_scaled.shape

---
## Part 2: PCA — The Linear Baseline

### TODO 2 — Fit PCA, inspect the scree plot, and project to 2D

**HINT:**
- Fit `PCA(random_state=RNG_SEED)` (no `n_components` limit yet) on `X_scaled` to get the full explained-variance spectrum for the scree plot
- Separately, fit `PCA(n_components=2, random_state=RNG_SEED)` for the 2D scatter
- Color the scatter by `df["segment"]` using `sns.scatterplot(..., hue=...)`

In [ ]:
# TODO: fit a full PCA to inspect explained variance
pca_full = ...
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, len(feature_cols) + 1), pca_full.explained_variance_ratio_)
axes[0].set(xlabel="Principal component", ylabel="Explained variance ratio", title="Scree Plot")
axes[1].plot(range(1, len(feature_cols) + 1), cumulative_variance, marker="o")
axes[1].axhline(0.9, color="red", linestyle="--", label="90% threshold")
axes[1].set(xlabel="Number of components", ylabel="Cumulative explained variance", title="Cumulative Variance")
axes[1].legend()
plt.tight_layout()
plt.show()

# TODO: fit PCA(n_components=2) and store the 2D projection
pca_2d = ...
X_pca_2d = ...  # pca_2d.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(7, 5.5))
sns.scatterplot(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1], hue=df["segment"].astype(str), palette="deep", alpha=0.6, s=20)
plt.title("PCA — 2D projection colored by K-Means segment")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Segment")
plt.show()

---
## Part 3: t-SNE — Local Structure, With Caveats

Remember from Week 3: t-SNE has **no `.transform()`** (every new dataset requires a full re-fit), inter-cluster *distances* and cluster *sizes* in the plot are not meaningful, and results vary run-to-run and with `perplexity`. We use it here purely as a second, non-linear opinion on separability — not as a component of any pipeline.

### TODO 3 — Fit t-SNE at two different perplexities and compare

**HINT:** `TSNE(n_components=2, perplexity=30, random_state=RNG_SEED, init="pca")` is a reasonable default; also try `perplexity=50` to see how much the picture changes. On 6000 points, this may take a little while to run — consider subsampling to ~2000 rows if it's too slow on your machine.

In [ ]:
# Optional: subsample for speed
tsne_sample_idx = np.random.RandomState(RNG_SEED).choice(len(X_scaled), size=2000, replace=False)
X_scaled_sample = X_scaled[tsne_sample_idx]
segment_sample = df["segment"].iloc[tsne_sample_idx].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, perplexity in zip(axes, [30, 50]):
    # TODO: fit TSNE(n_components=2, perplexity=perplexity, random_state=RNG_SEED, init="pca") on X_scaled_sample
    tsne = ...
    X_tsne = ...  # tsne.fit_transform(X_scaled_sample)
    sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=segment_sample, palette="deep", alpha=0.6, s=20, ax=ax, legend=(perplexity == 30))
    ax.set_title(f"t-SNE (perplexity={perplexity})")
plt.tight_layout()
plt.show()

---
## Part 4: UMAP — Faster, and Supports `.transform()` on New Customers

Unlike t-SNE, a fitted UMAP model *can* embed brand-new customers without recomputing the whole layout — relevant if this project ever needed a "live" 2D dashboard that updates as new customers sign up.

### TODO 4 — Fit UMAP (skipped gracefully if not installed)

**HINT:** `umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RNG_SEED)`, then `.fit_transform(X_scaled)`.

In [ ]:
if UMAP_AVAILABLE:
    # TODO: fit umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RNG_SEED) on X_scaled
    umap_model = ...
    X_umap = ...  # umap_model.fit_transform(X_scaled)

    plt.figure(figsize=(7, 5.5))
    sns.scatterplot(x=X_umap[:, 0], y=X_umap[:, 1], hue=df["segment"].astype(str), palette="deep", alpha=0.6, s=20)
    plt.title("UMAP — 2D projection colored by K-Means segment")
    plt.legend(title="Segment")
    plt.show()
else:
    print("Skipped — install umap-learn to run this cell.")

---
## Part 5: Decision Framework — Which Technique for Which Audience?

| Question | Best choice |
|---|---|
| "I need to explain *why* two customers are far apart" | PCA — axes are linear combinations of real features, at least somewhat interpretable |
| "I want the most visually crisp separation for a stakeholder slide" | t-SNE or UMAP, usually |
| "A new customer signs up daily and I need to place them on the existing map" | UMAP (`.transform()` on new data) — t-SNE cannot do this at all |
| "I need this to run in under a second on 100k+ rows" | PCA, then UMAP; t-SNE is the slowest of the three |
| "I want a feature to feed into the churn model" | **None of them** — see Senior Engineer Notes below |

---
## Senior Engineer Notes & Best Practices

1. **Reuse fitted preprocessing artefacts across notebooks.** Refitting a new `StandardScaler` here — even on the same raw data — would technically produce very similar numbers, but in a real pipeline where data changes over time, it would silently desynchronize this visualization from the actual segmentation being described.
2. **PCA's linearity is a feature, not just a limitation.** When a stakeholder asks "what does this axis mean?", PCA is the only one of the three you can answer for (approximately, via component loadings) — t-SNE/UMAP axes have no such interpretation.
3. **Never skip stating the perplexity/`n_neighbors` value when sharing a t-SNE/UMAP plot** — the exact same data can look like 2 clusters or 6 depending on this hyperparameter, and an unlabeled plot is unreproducible and easy to misread.
4. **Dimensionality reduction for visualization and dimensionality reduction for modeling are different jobs** — this project deliberately never lets a PCA/UMAP component become a churn-model input, precisely to keep every model feature traceable to a real, explainable business quantity.
5. **Cross-check the DR plot against the *known* algorithm output, not the other way around.** If the 2D plot happens to show 5 visually distinct blobs but K-Means was run with $k=4$, that's a prompt to revisit Notebook 1's $k$ selection — not a reason to silently trust the picture over the metric.

## Key Takeaways
- Always transform with *already-fitted* preprocessing objects when visualizing an existing model's output — never refit
- PCA (linear, interpretable, fast) → t-SNE (best local structure, no `.transform()`, slow) → UMAP (fast, supports `.transform()`, weaker interpretability) — pick based on the audience and the constraint that matters most
- This project's segmentation and churn features stay in fully interpretable, real-business-quantity space; PCA/t-SNE/UMAP live only in the visualization layer, never inside the modeling pipeline